# Part 3: Decision Engine
## Win Rate Driver Analysis System

---

### Business Objective

Build a **driver decomposition system** that quantifies the impact of each factor on win rate, enabling the CRO to:
1. Understand exactly which factors are hurting/helping performance
2. Take targeted, prioritized actions
3. Track improvement over time

---

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load and prepare data
df = pd.read_csv('skygeni_sales_data.csv')
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])
df['closed_quarter'] = df['closed_date'].dt.to_period('Q')
df['closed_month'] = df['closed_date'].dt.to_period('M')
df['is_won'] = (df['outcome'] == 'Won').astype(int)

# Create derived features
df['deal_size_bucket'] = pd.cut(df['deal_amount'], 
                                 bins=[0, 10000, 25000, 50000, 100000, np.inf],
                                 labels=['<$10K', '$10K-25K', '$25K-50K', '$50K-100K', '$100K+'])

df['cycle_bucket'] = pd.cut(df['sales_cycle_days'],
                            bins=[0, 30, 60, 90, 120, np.inf],
                            labels=['Fast (0-30)', 'Normal (31-60)', 'Slow (61-90)', 'Very Slow (91-120)', 'Stalled (120+)'])

print(f"✅ Loaded {len(df):,} deals")
print(f"📅 Date range: {df['closed_date'].min().strftime('%Y-%m-%d')} to {df['closed_date'].max().strftime('%Y-%m-%d')}")

In [ ]:
# Define analysis period (last two complete quarters)
ANALYSIS_QUARTERS = ['2024Q1', '2024Q2']
df_analysis = df[df['closed_quarter'].astype(str).isin(ANALYSIS_QUARTERS)].copy()

# Calculate baseline metrics
OVERALL_WIN_RATE = df_analysis['is_won'].mean()
TOTAL_DEALS = len(df_analysis)

print("=" * 60)
print("ANALYSIS PERIOD: Q1-Q2 2024")
print("=" * 60)
print(f"\n📊 Total Deals: {TOTAL_DEALS:,}")
print(f"📊 Overall Win Rate: {OVERALL_WIN_RATE*100:.1f}%")
print(f"📊 Target Win Rate (Q4 2023 benchmark): 47.5%")
print(f"📊 Gap to Target: {(0.475 - OVERALL_WIN_RATE)*100:.1f}pp")

---

## 1. Driver Impact Framework

### Methodology

We calculate the **Driver Impact Score (DIS)** for each factor using:

```
Driver Impact Score = (Factor Win Rate - Overall Win Rate) × Volume Weight × 1000

Where:
- Factor Win Rate = Win rate for that specific segment
- Overall Win Rate = Baseline (45.5% for Q1-Q2 2024)
- Volume Weight = Segment's proportion of total deals
- ×1000 = Scaling for readability
```

**Interpretation:**
- Positive DIS = This factor is **HELPING** win rate
- Negative DIS = This factor is **HURTING** win rate
- Magnitude indicates the SIZE of impact

In [ ]:
def calculate_driver_impact(df_period, factor_col, baseline_rate):
    """
    Calculate Driver Impact Score for each level of a factor.
    
    Returns DataFrame with:
    - factor: The factor level (e.g., 'Partner', 'HealthTech')
    - win_rate: Win rate for this factor level
    - deal_count: Number of deals
    - volume_pct: Percentage of total deals
    - delta_pp: Difference from baseline in percentage points
    - impact_score: Driver Impact Score (scaled)
    """
    
    # Calculate metrics by factor level
    result = df_period.groupby(factor_col).agg(
        deal_count=('deal_id', 'count'),
        wins=('is_won', 'sum'),
        win_rate=('is_won', 'mean'),
        avg_value=('deal_amount', 'mean'),
        avg_cycle=('sales_cycle_days', 'mean')
    ).reset_index()
    
    result.columns = ['factor', 'deal_count', 'wins', 'win_rate', 'avg_value', 'avg_cycle']
    
    # Calculate derived metrics
    total_deals = result['deal_count'].sum()
    result['volume_pct'] = result['deal_count'] / total_deals * 100
    result['delta_pp'] = (result['win_rate'] - baseline_rate) * 100
    
    # Driver Impact Score = Delta × Volume Weight × Scale Factor
    result['impact_score'] = result['delta_pp'] * (result['volume_pct'] / 100) * 10
    
    # Win rate as percentage
    result['win_rate_pct'] = result['win_rate'] * 100
    
    return result.sort_values('impact_score')

print("✅ Driver Impact Function defined")

In [ ]:
# Calculate Driver Impact for all dimensions
driver_dimensions = [
    ('lead_source', 'Lead Source'),
    ('industry', 'Industry'),
    ('region', 'Region'),
    ('product_type', 'Product Type'),
    ('sales_rep_id', 'Sales Rep'),
    ('deal_size_bucket', 'Deal Size'),
    ('cycle_bucket', 'Sales Cycle'),
    ('deal_stage', 'Entry Stage')
]

all_drivers = {}
for col, name in driver_dimensions:
    all_drivers[name] = calculate_driver_impact(df_analysis, col, OVERALL_WIN_RATE)
    
print("✅ All driver impacts calculated")

---

## 2. Driver Scorecard

A unified view of all factors ranked by their impact on win rate.

In [ ]:
# Create unified driver scorecard
scorecard_rows = []

for dim_name, driver_df in all_drivers.items():
    for _, row in driver_df.iterrows():
        scorecard_rows.append({
            'dimension': dim_name,
            'factor': row['factor'],
            'win_rate': row['win_rate_pct'],
            'deals': row['deal_count'],
            'volume_pct': row['volume_pct'],
            'delta_pp': row['delta_pp'],
            'impact_score': row['impact_score']
        })

scorecard = pd.DataFrame(scorecard_rows)

# Exclude individual reps from the main scorecard (too granular)
scorecard_main = scorecard[scorecard['dimension'] != 'Sales Rep'].copy()
scorecard_reps = scorecard[scorecard['dimension'] == 'Sales Rep'].copy()

print("=" * 70)
print("📊 DRIVER SCORECARD - Factors Impacting Win Rate")
print("=" * 70)
print(f"\nBaseline Win Rate: {OVERALL_WIN_RATE*100:.1f}%")
print(f"\n{'='*70}")
print("\n🔴 TOP NEGATIVE DRIVERS (Hurting Win Rate):")
print("-" * 70)

negative = scorecard_main[scorecard_main['impact_score'] < 0].nsmallest(10, 'impact_score')
print(f"{'Dimension':<15} {'Factor':<20} {'Win Rate':>10} {'Δ vs Avg':>10} {'Impact':>10}")
print("-" * 70)
for _, row in negative.iterrows():
    print(f"{row['dimension']:<15} {str(row['factor']):<20} {row['win_rate']:>9.1f}% {row['delta_pp']:>+9.1f}pp {row['impact_score']:>+9.2f}")

In [ ]:
print("\n🟢 TOP POSITIVE DRIVERS (Helping Win Rate):")
print("-" * 70)

positive = scorecard_main[scorecard_main['impact_score'] > 0].nlargest(10, 'impact_score')
print(f"{'Dimension':<15} {'Factor':<20} {'Win Rate':>10} {'Δ vs Avg':>10} {'Impact':>10}")
print("-" * 70)
for _, row in positive.iterrows():
    print(f"{row['dimension']:<15} {str(row['factor']):<20} {row['win_rate']:>9.1f}% {row['delta_pp']:>+9.1f}pp {row['impact_score']:>+9.2f}")

In [ ]:
# Visualize driver impacts by dimension
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

dims_to_plot = ['Lead Source', 'Industry', 'Region', 'Product Type', 'Deal Size', 'Entry Stage']

for ax, dim_name in zip(axes, dims_to_plot):
    data = all_drivers[dim_name].sort_values('impact_score')
    colors = ['#DC3545' if x < 0 else '#28A745' for x in data['impact_score']]
    
    bars = ax.barh(data['factor'].astype(str), data['impact_score'], color=colors, alpha=0.8)
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_xlabel('Driver Impact Score')
    ax.set_title(f'{dim_name}', fontweight='bold')
    
    # Add win rate labels
    for bar, wr in zip(bars, data['win_rate_pct']):
        x_pos = bar.get_width() + 0.1 if bar.get_width() >= 0 else bar.get_width() - 0.1
        ha = 'left' if bar.get_width() >= 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{wr:.1f}%', va='center', ha=ha, fontsize=9)

plt.suptitle('📊 Driver Impact Analysis by Dimension', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Interaction Analysis: Toxic & Golden Combinations

Some factor combinations perform much better or worse than individual factors suggest. Let's find these **synergies and conflicts**.

In [ ]:
# Analyze two-way interactions
def analyze_interactions(df_period, factor1, factor2, min_deals=20):
    """
    Analyze win rate for combinations of two factors.
    Returns combinations sorted by deviation from expected.
    """
    
    # Overall win rate for each factor level
    f1_rates = df_period.groupby(factor1)['is_won'].mean().to_dict()
    f2_rates = df_period.groupby(factor2)['is_won'].mean().to_dict()
    
    # Combination analysis
    combo_stats = df_period.groupby([factor1, factor2]).agg(
        deals=('deal_id', 'count'),
        wins=('is_won', 'sum'),
        actual_wr=('is_won', 'mean')
    ).reset_index()
    
    # Filter for sufficient sample size
    combo_stats = combo_stats[combo_stats['deals'] >= min_deals]
    
    # Calculate expected win rate (average of individual factor rates)
    combo_stats['expected_wr'] = combo_stats.apply(
        lambda r: (f1_rates[r[factor1]] + f2_rates[r[factor2]]) / 2, axis=1
    )
    
    # Deviation from expected
    combo_stats['deviation'] = (combo_stats['actual_wr'] - combo_stats['expected_wr']) * 100
    
    # Label combinations
    combo_stats['combo'] = combo_stats[factor1].astype(str) + ' + ' + combo_stats[factor2].astype(str)
    
    return combo_stats.sort_values('deviation')

# Key interaction: Industry × Lead Source
interactions_ind_lead = analyze_interactions(df_analysis, 'industry', 'lead_source')

print("=" * 70)
print("📊 INTERACTION ANALYSIS: Industry × Lead Source")
print("=" * 70)

print("\n🔴 TOXIC COMBINATIONS (Worse than expected):")
print("-" * 70)
print(f"{'Combination':<35} {'Deals':>8} {'Actual':>10} {'Expected':>10} {'Gap':>10}")
print("-" * 70)
for _, row in interactions_ind_lead.head(5).iterrows():
    print(f"{row['combo']:<35} {row['deals']:>8} {row['actual_wr']*100:>9.1f}% {row['expected_wr']*100:>9.1f}% {row['deviation']:>+9.1f}pp")

print("\n🟢 GOLDEN COMBINATIONS (Better than expected):")
print("-" * 70)
for _, row in interactions_ind_lead.tail(5).iloc[::-1].iterrows():
    print(f"{row['combo']:<35} {row['deals']:>8} {row['actual_wr']*100:>9.1f}% {row['expected_wr']*100:>9.1f}% {row['deviation']:>+9.1f}pp")

In [ ]:
# Interaction: Product Type × Region
interactions_prod_reg = analyze_interactions(df_analysis, 'product_type', 'region')

print("\n" + "=" * 70)
print("📊 INTERACTION ANALYSIS: Product Type × Region")
print("=" * 70)

print("\n🔴 TOXIC:")
for _, row in interactions_prod_reg.head(3).iterrows():
    print(f"   {row['combo']}: {row['actual_wr']*100:.1f}% (expected {row['expected_wr']*100:.1f}%, gap {row['deviation']:+.1f}pp)")

print("\n🟢 GOLDEN:")
for _, row in interactions_prod_reg.tail(3).iloc[::-1].iterrows():
    print(f"   {row['combo']}: {row['actual_wr']*100:.1f}% (expected {row['expected_wr']*100:.1f}%, gap {row['deviation']:+.1f}pp)")

In [ ]:
# Create heatmap of Industry × Lead Source
pivot = df_analysis.pivot_table(
    values='is_won',
    index='industry',
    columns='lead_source',
    aggfunc='mean'
) * 100

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=OVERALL_WIN_RATE*100,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Win Rate (%)'})
ax.set_title('📊 Win Rate: Industry × Lead Source Interaction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. Sales Rep Driver Analysis

Which reps are succeeding/struggling with which segments?

In [ ]:
# Rep performance summary
rep_summary = scorecard_reps.sort_values('impact_score')

print("=" * 70)
print("📊 SALES REP DRIVER ANALYSIS")
print("=" * 70)

# Categorize reps
rep_summary['category'] = pd.cut(
    rep_summary['delta_pp'],
    bins=[-100, -5, 5, 100],
    labels=['🔴 Below Average', '🟡 On Target', '🟢 Above Average']
)

print("\n📊 Rep Distribution:")
print(rep_summary['category'].value_counts())

print("\n🔴 BOTTOM 5 REPS (Need Coaching):")
print("-" * 50)
for _, row in rep_summary.head(5).iterrows():
    print(f"   {row['factor']}: {row['win_rate']:.1f}% win rate ({row['delta_pp']:+.1f}pp vs avg), {int(row['deals'])} deals")

print("\n🟢 TOP 5 REPS (Best Practices):")
print("-" * 50)
for _, row in rep_summary.tail(5).iloc[::-1].iterrows():
    print(f"   {row['factor']}: {row['win_rate']:.1f}% win rate ({row['delta_pp']:+.1f}pp vs avg), {int(row['deals'])} deals")

In [ ]:
# Rep × Industry fit analysis (who should sell to whom?)
rep_industry = df_analysis.groupby(['sales_rep_id', 'industry']).agg(
    deals=('deal_id', 'count'),
    wins=('is_won', 'sum'),
    win_rate=('is_won', 'mean')
).reset_index()

# Filter for meaningful sample
rep_industry = rep_industry[rep_industry['deals'] >= 5]

# Find best rep for each industry
print("\n📊 BEST REP FOR EACH INDUSTRY:")
print("-" * 60)
best_by_industry = rep_industry.loc[rep_industry.groupby('industry')['win_rate'].idxmax()]
for _, row in best_by_industry.iterrows():
    print(f"   {row['industry']}: {row['sales_rep_id']} ({row['win_rate']*100:.1f}% on {row['deals']} deals)")

# Find concerning mismatches (low volume reps in key industries)
print("\n⚠️ REP-INDUSTRY MISMATCHES (Low performance, high volume):")
print("-" * 60)
concerning = rep_industry[(rep_industry['win_rate'] < 0.40) & (rep_industry['deals'] >= 10)].sort_values('win_rate')
for _, row in concerning.head(5).iterrows():
    print(f"   {row['sales_rep_id']} + {row['industry']}: {row['win_rate']*100:.1f}% on {row['deals']} deals")

---

## 5. Action Recommendation Engine

Generate specific, prioritized recommendations for the CRO.

In [ ]:
# Generate recommendations based on analysis
print("=" * 70)
print("🎯 ACTION RECOMMENDATIONS")
print("=" * 70)

recommendations = []

# 1. Lead Source recommendations
worst_lead = all_drivers['Lead Source'].iloc[0]
best_lead = all_drivers['Lead Source'].iloc[-1]

if worst_lead['delta_pp'] < -2:
    recommendations.append({
        'priority': 1,
        'category': 'Lead Quality',
        'action': f"Audit {worst_lead['factor']} lead channel",
        'detail': f"Win rate is {worst_lead['win_rate_pct']:.1f}% ({worst_lead['delta_pp']:+.1f}pp vs avg). Review lead qualification criteria for this source.",
        'impact': f"Affects {worst_lead['volume_pct']:.1f}% of pipeline ({int(worst_lead['deal_count'])} deals)"
    })

# 2. Industry recommendations
worst_industry = all_drivers['Industry'].iloc[0]
if worst_industry['delta_pp'] < -2:
    recommendations.append({
        'priority': 2,
        'category': 'ICP Focus',
        'action': f"Review {worst_industry['factor']} vertical strategy",
        'detail': f"Win rate dropped to {worst_industry['win_rate_pct']:.1f}%. Consider vertical specialization or adjusted messaging.",
        'impact': f"Affects {worst_industry['volume_pct']:.1f}% of pipeline"
    })

# 3. Rep coaching recommendations
bottom_reps = scorecard_reps.nsmallest(3, 'delta_pp')
rep_list = ', '.join(bottom_reps['factor'].tolist())
recommendations.append({
    'priority': 3,
    'category': 'Sales Coaching',
    'action': f"Prioritize coaching for {rep_list}",
    'detail': f"These reps are {abs(bottom_reps['delta_pp'].mean()):.1f}pp below average. Pair with top performers for shadowing.",
    'impact': f"Combined {int(bottom_reps['deals'].sum())} deals affected"
})

# 4. Golden combination recommendations
best_combo = interactions_ind_lead.tail(1).iloc[0]
recommendations.append({
    'priority': 4,
    'category': 'Lead Routing',
    'action': f"Increase focus on {best_combo['combo']}",
    'detail': f"This combination has {best_combo['actual_wr']*100:.1f}% win rate ({best_combo['deviation']:+.1f}pp above expected).",
    'impact': f"Currently {int(best_combo['deals'])} deals - room to grow"
})

# 5. Toxic combination warning
worst_combo = interactions_ind_lead.head(1).iloc[0]
recommendations.append({
    'priority': 5,
    'category': 'Risk Avoidance',
    'action': f"Add friction for {worst_combo['combo']}",
    'detail': f"This combination has only {worst_combo['actual_wr']*100:.1f}% win rate. Require extra qualification or reassign to specialists.",
    'impact': f"{int(worst_combo['deals'])} deals at risk"
})

# Display recommendations
print("\n" + "━" * 70)
for rec in recommendations:
    print(f"\n📌 PRIORITY {rec['priority']}: {rec['category'].upper()}")
    print(f"   Action: {rec['action']}")
    print(f"   Detail: {rec['detail']}")
    print(f"   Impact: {rec['impact']}")

In [ ]:
# Generate "This Week" focus summary
print("\n" + "=" * 70)
print("📋 THIS WEEK'S TOP 3 FOCUS AREAS")
print("=" * 70)

top3 = recommendations[:3]
for i, rec in enumerate(top3, 1):
    print(f"\n{i}. {rec['action']}")
    print(f"   → {rec['detail']}")

---

## 6. How a Sales Leader Would Use This

### Weekly Pipeline Review (30 min)

1. **Check the Driver Scorecard** (2 min)
   - Glance at top negative drivers to see what's hurting this week
   - Note any changes from last week

2. **Review Top 3 Actions** (5 min)
   - Discuss progress on last week's actions
   - Assign owners for this week's priorities

3. **Rep Coaching Prep** (10 min)
   - Identify 1-2 reps for focused 1:1s
   - Pull their deal mix to understand if it's a coaching or assignment issue

4. **Deal Routing Decisions** (10 min)
   - Flag any new deals matching toxic combinations
   - Route golden combination deals to appropriate reps

### Monthly Strategic Review

1. **ICP Refinement** - Update ideal customer profile based on industry/lead source performance
2. **Territory Optimization** - Adjust rep-segment assignments based on fit analysis
3. **Marketing Alignment** - Share lead source quality data with marketing team

### Quarterly Planning

1. **Hiring Decisions** - Identify which segments need more coverage
2. **Training Investment** - Target training programs based on driver gaps
3. **Channel Strategy** - Reallocate budget to higher-performing lead sources

In [ ]:
# Export summary for dashboard
summary_export = {
    'analysis_period': 'Q1-Q2 2024',
    'total_deals': TOTAL_DEALS,
    'overall_win_rate': f"{OVERALL_WIN_RATE*100:.1f}%",
    'target_win_rate': '47.5%',
    'gap_to_target': f"{(0.475 - OVERALL_WIN_RATE)*100:.1f}pp",
    'top_negative_driver': f"{scorecard_main.iloc[0]['dimension']}: {scorecard_main.iloc[0]['factor']}",
    'top_positive_driver': f"{scorecard_main.iloc[-1]['dimension']}: {scorecard_main.iloc[-1]['factor']}",
    'reps_below_average': len(scorecard_reps[scorecard_reps['delta_pp'] < -5]),
    'reps_above_average': len(scorecard_reps[scorecard_reps['delta_pp'] > 5])
}

print("\n" + "=" * 70)
print("📊 EXECUTIVE SUMMARY")
print("=" * 70)
for key, value in summary_export.items():
    print(f"   {key.replace('_', ' ').title()}: {value}")

---

## Conclusion

This **Win Rate Driver Analysis System** provides:

| Component | Purpose | Key Output |
|-----------|---------|------------|
| **Driver Scorecard** | Unified view of all factors | Ranked list by impact |
| **Interaction Analysis** | Find synergies & conflicts | Toxic/golden combinations |
| **Rep-Segment Fit** | Optimize assignments | Best rep for each vertical |
| **Action Recommendations** | Specific next steps | Top 3 weekly priorities |

### Design Choices Made:

1. **Rule-based over ML** - More interpretable for sales leaders
2. **Impact Score** - Combines rate delta with volume for prioritization
3. **Expected vs Actual** - Identifies true synergies beyond individual factors
4. **Actionable outputs** - Every insight maps to a specific action

### What This Does NOT Do:

- Predict individual deal outcomes (that's Option A)
- Forecast revenue (that's Option C)
- Detect anomalies (that's Option D)

It focuses solely on **explaining WHY** win rates are what they are and **WHAT TO DO ABOUT IT**.

---

# 📋 Assignment Requirements: How This Solution Addresses the Rubric

---

## 1. Problem Definition

### The Problem

**Business Challenge:** The CRO reports that win rate has dropped over the last two quarters, but doesn't know *why* or *where to focus* improvement efforts.

**Technical Translation:** We need a system that:
1. Decomposes overall win rate into contributions from individual factors
2. Quantifies the impact of each factor (not just rate, but weighted by volume)
3. Identifies interaction effects that aren't visible in single-factor analysis
4. Generates specific, prioritized recommendations

### Why Win Rate Driver Analysis (Option B)?

| Option | What It Does | Why We Chose/Didn't Choose |
|--------|-------------|---------------------------|
| **A: Deal Risk Scoring** | Predicts which deals will be lost | Useful but *reactive* - doesn't explain *why* |
| **B: Win Rate Driver Analysis** ✅ | Explains what's hurting/helping win rate | *Proactive* - enables systemic fixes |
| **C: Revenue Forecast** | Predicts next quarter's revenue | *Forecasting* - doesn't guide action |
| **D: Anomaly Detection** | Flags unusual patterns | *Monitoring* - but doesn't explain causality |

**We chose Option B** because it directly answers the CRO's question: *"What exactly is going wrong?"* Once we know the drivers, interventions become obvious.

## 2. Model/Rule-Based System

### Modeling Choices

We implemented a **rule-based driver decomposition system** rather than a machine learning model. Here's why:

| Aspect | ML Approach | Our Rule-Based Approach |
|--------|-------------|------------------------|
| **Interpretability** | Black box (SHAP helps but adds complexity) | Fully transparent calculations |
| **Data Requirements** | Needs more data for training | Works with current dataset |
| **Maintenance** | Requires retraining | Simple updates to thresholds |
| **Trust** | Sales leaders skeptical of "AI" | Can validate calculations manually |
| **Speed to Value** | Weeks of development | Immediate insights |

### Core Algorithm: Driver Impact Score

```python
Driver Impact Score = (Factor Win Rate - Overall Win Rate) × Volume Weight × Scale
```

**Design Rationale:**
- **Delta from baseline:** Shows if factor is above/below average
- **Volume weighting:** A small segment with bad rate matters less than a large one
- **Scale factor:** Makes scores readable (avoids tiny decimals)

### Interaction Analysis: Expected vs Actual

```python
Expected Win Rate = (Factor1 Rate + Factor2 Rate) / 2
Deviation = Actual Rate - Expected Rate
```

**Design Rationale:**
- Simple averaging gives a baseline expectation
- Positive deviation = synergy (factors work well together)
- Negative deviation = conflict (combination is toxic)

### Why Not More Sophisticated Methods?

1. **Regression coefficients:** Harder to explain, assumes linearity
2. **Decision trees:** Good for prediction, not for impact quantification
3. **Causal inference:** Requires assumptions about confounders that aren't defensible here

**Our approach prioritizes business usefulness over statistical sophistication.**

## 3. Actionable Outputs

### Output 1: Driver Scorecard

**What it is:** Ranked list of all factors by positive/negative impact on win rate.

**How it's actionable:**
- Top negative drivers = immediate focus areas
- Top positive drivers = areas to double down on
- Clear prioritization by magnitude

---

### Output 2: Toxic/Golden Combinations

**What it is:** Factor combinations that perform better or worse than expected.

**How it's actionable:**
- **Toxic combos:** Add qualification friction, route to specialists, or deprioritize
- **Golden combos:** Increase investment, create playbooks, assign top reps

---

### Output 3: Rep Performance Analysis

**What it is:** Which reps are above/below average and their segment fit.

**How it's actionable:**
- Identify coaching targets (bottom performers)
- Identify best practice sources (top performers)
- Optimize rep-segment assignments

---

### Output 4: Weekly Recommendations

**What it is:** Top 3 prioritized actions for this week.

**How it's actionable:**
- Ready for Monday pipeline review
- Specific enough to assign owners
- Tied to quantified impact

## 4. How a Sales Leader Would Use This

### Weekly Pipeline Review (30 minutes)

| Time | Activity | This System Helps By... |
|------|----------|------------------------|
| 0-5 min | Review driver scorecard | Seeing what's hurting win rate this week |
| 5-15 min | Discuss top 3 recommendations | Having pre-prioritized action items |
| 15-25 min | Rep performance check | Identifying who needs 1:1 coaching |
| 25-30 min | Assign actions | Clear ownership with quantified stakes |

### 1:1 Coaching Sessions

**Before the meeting:**
- Check rep's Driver Impact Score vs peers
- Review their segment mix (are they getting hard deals?)
- Look at their toxic combo exposure

**During the meeting:**
- Share specific improvement areas with data
- Connect them with top performers in similar segments
- Set measurable goals based on driver gaps

### Strategic Planning (Monthly/Quarterly)

| Decision | How This System Informs It |
|----------|---------------------------|
| **ICP refinement** | Which industries/segments should we target? |
| **Channel investment** | Which lead sources deserve more budget? |
| **Hiring priorities** | Which segments need more/better coverage? |
| **Training focus** | What skills gaps are hurting specific verticals? |
| **Product packaging** | Is Enterprise underperforming vs Core/Pro? |

### Real Example Usage

**Scenario:** CRO sees EdTech is a negative driver (-6.27 impact)

**Actions they could take:**
1. Check if EdTech budget cycles align with our sales timeline
2. Review competitive wins/losses in EdTech
3. Assign EdTech deals to reps with vertical experience
4. Consider a vertical-specific sales deck
5. If all else fails, deprioritize EdTech in lead scoring

## 5. Design Philosophy

### What We Optimized For

| Priority | Why It Matters |
|----------|---------------|
| **Interpretability** | A model no one trusts is useless |
| **Actionability** | Insights without actions are just noise |
| **Speed to insight** | Weekly cadence beats quarterly analysis |
| **Business language** | No p-values or confidence intervals for sales leaders |

### What We Did NOT Optimize For

| Deprioritized | Why |
|---------------|-----|
| **Kaggle-style accuracy** | This isn't a prediction competition |
| **Novel algorithms** | Proven methods applied well beat complex models applied poorly |
| **Exhaustive feature engineering** | Diminishing returns for interpretability cost |

### Key Trade-offs Made

1. **Simple averaging for expected rates** vs. regression-based expected values
   - *Trade-off:* Less precise but more explainable

2. **Rule-based impact scores** vs. SHAP values from a classifier
   - *Trade-off:* Less theoretically grounded but immediately understandable

3. **Weekly snapshot** vs. rolling window analysis
   - *Trade-off:* Simpler to implement and explain

---

**Bottom Line:** This system is designed to be *used*, not just *admired*. Every output should drive a conversation or decision.